# Recommendation Engine — Thu nghiem offline
Thu nghiem logic goi y san pham truoc khi dua vao `app/models/recommendation.py`.

**Cac loai goi y:**
1. Popular products (view count)
2. Content-based: san pham tuong tu (cosine similarity)
3. Collaborative: theo user history (weighted embedding)
4. Keyword-based


In [ ]:
import numpy as np
import pickle, json, os
from datetime import datetime, timedelta
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

np.random.seed(42)
print("Ready")


## 1. Mock data

In [ ]:
PRODUCTS = [
    {"id":"p1","name":"Noi com Panasonic 1.8L","cat":"nau_nuong","price":1200000},
    {"id":"p2","name":"Noi com Toshiba 1.0L","cat":"nau_nuong","price":800000},
    {"id":"p3","name":"Chao chong dinh Sunhouse 28cm","cat":"chao_noi","price":350000},
    {"id":"p4","name":"Chao gang Lodge 26cm","cat":"chao_noi","price":950000},
    {"id":"p5","name":"Dao bep Nhat Kai 18cm","cat":"dao_dung_cu","price":450000},
    {"id":"p6","name":"Bo dao nha bep 5 mon","cat":"dao_dung_cu","price":620000},
    {"id":"p7","name":"May xay sinh to Philips 750W","cat":"may_xay_ep","price":1500000},
    {"id":"p8","name":"May ep cham Hurom H-AA","cat":"may_xay_ep","price":4200000},
    {"id":"p9","name":"Lo nuong Electrolux 38L","cat":"lo_nuong","price":3100000},
    {"id":"p10","name":"Noi ap suat Supor 5L","cat":"nau_nuong","price":890000},
]

VIEW_LOG = [
    {"userId":"u1","productId":"p1","createdAt":datetime.now()-timedelta(days=1)},
    {"userId":"u1","productId":"p2","createdAt":datetime.now()-timedelta(days=2)},
    {"userId":"u1","productId":"p3","createdAt":datetime.now()-timedelta(days=1)},
    {"userId":"u2","productId":"p3","createdAt":datetime.now()-timedelta(days=3)},
    {"userId":"u2","productId":"p4","createdAt":datetime.now()-timedelta(days=2)},
    {"userId":"u2","productId":"p7","createdAt":datetime.now()-timedelta(days=1)},
    {"userId":"u3","productId":"p5","createdAt":datetime.now()-timedelta(days=5)},
    {"userId":"u3","productId":"p6","createdAt":datetime.now()-timedelta(days=4)},
    {"userId":"u1","productId":"p1","createdAt":datetime.now()-timedelta(hours=3)},
    {"userId":"u1","productId":"p10","createdAt":datetime.now()-timedelta(hours=1)},
]

USER_HISTORY = {
    "u1": [
        {"productId":"p1","type":"purchase","timestamp":datetime.now()-timedelta(days=10)},
        {"productId":"p2","type":"view","timestamp":datetime.now()-timedelta(days=2)},
        {"productId":"p3","type":"add_to_cart","timestamp":datetime.now()-timedelta(days=1)},
        {"productId":"p10","type":"view","timestamp":datetime.now()-timedelta(hours=1)},
    ],
    "u2": [
        {"productId":"p7","type":"purchase","timestamp":datetime.now()-timedelta(days=5)},
        {"productId":"p3","type":"view","timestamp":datetime.now()-timedelta(days=3)},
        {"productId":"p4","type":"add_to_wishlist","timestamp":datetime.now()-timedelta(days=2)},
    ],
    "u3": [
        {"productId":"p5","type":"purchase","timestamp":datetime.now()-timedelta(days=7)},
        {"productId":"p6","type":"purchase","timestamp":datetime.now()-timedelta(days=3)},
    ],
}

prod_map = {p["id"]:p for p in PRODUCTS}
print(len(PRODUCTS), "san pham,", len(USER_HISTORY), "users")


## 2. Build product embeddings

In [ ]:
texts = [p["name"]+" "+p["cat"] for p in PRODUCTS]
ids   = [p["id"] for p in PRODUCTS]
vec = TfidfVectorizer(ngram_range=(1,2))
emb_mat = vec.fit_transform(texts).toarray()
product_embeddings = {pid: emb_mat[i] for i,pid in enumerate(ids)}
print("Embedding matrix:", emb_mat.shape)


## 3. Popular products — view count

In [ ]:
def get_popular(limit=5):
    counts = Counter(v["productId"] for v in VIEW_LOG)
    print("Popular:")
    for pid, cnt in counts.most_common(limit):
        print("  %s views | %s" % (cnt, prod_map[pid]["name"]))

get_popular()


## 4. Content-based — san pham tuong tu

In [ ]:
sim_mat = cosine_similarity(emb_mat)

def get_similar(qid, k=3):
    i = ids.index(qid)
    ranked = np.argsort(sim_mat[i])[::-1]
    print("Tuong tu %s:" % prod_map[qid]["name"])
    for ri in ranked[1:k+1]:
        print("  %.3f  %s" % (sim_mat[i][ri], PRODUCTS[ri]["name"]))

get_similar("p1"); print()
get_similar("p5"); print()
get_similar("p7")


## 5. Interaction weights — mirror production logic

In [ ]:
# Mirror _calculate_interaction_weight() in recommendation.py
TYPE_W = {"view":1.0, "add_to_cart":2.0, "add_to_wishlist":2.5, "purchase":5.0}

def interaction_weight(item):
    base = TYPE_W.get(item["type"], 1.0)
    days = (datetime.now() - item["timestamp"]).days
    recency = max(0.5, min(1.0, 1.0 - days/30))
    return base * recency

print("u1 weights:")
for item in USER_HISTORY["u1"]:
    w = interaction_weight(item)
    name = prod_map[item["productId"]]["name"][:28]
    print("  %-15s %-28s -> %.2f" % (item["type"], name, w))


## 6. Collaborative filtering — user embedding

In [ ]:
def build_user_emb(uid):
    hist = USER_HISTORY.get(uid, [])
    dim = emb_mat.shape[1]
    u = np.zeros(dim)
    total_w = 0
    for item in hist:
        pid = item["productId"]
        if pid in product_embeddings:
            w = interaction_weight(item)
            u += w * product_embeddings[pid]
            total_w += w
    return u / total_w if total_w > 0 else None

user_embeddings = {uid: build_user_emb(uid) for uid in USER_HISTORY}
print("User embeddings:", list(user_embeddings.keys()))


In [ ]:
def get_personalized(uid, k=5, exclude_viewed=True):
    if uid not in user_embeddings or user_embeddings[uid] is None:
        print("No emb for", uid, "- fallback to popular")
        return get_popular(k)
    viewed = set()
    if exclude_viewed:
        viewed = {it["productId"] for it in USER_HISTORY.get(uid, [])}
    u_emb = user_embeddings[uid]
    scores = [(pid, cosine_similarity([u_emb],[pe])[0][0])
              for pid,pe in product_embeddings.items() if pid not in viewed]
    scores.sort(key=lambda x: x[1], reverse=True)
    print("Goi y cho %s:" % uid)
    for pid, s in scores[:k]:
        print("  %.3f  %s" % (s, prod_map[pid]["name"]))
    return scores[:k]

get_personalized("u1"); print()
get_personalized("u2")


## 7. Keyword-based

In [ ]:
def get_by_keyword(keywords, k=4):
    q = vec.transform([" ".join(keywords)]).toarray()[0]
    scores = [(pid, cosine_similarity([q],[pe])[0][0])
              for pid,pe in product_embeddings.items()]
    scores.sort(key=lambda x: x[1], reverse=True)
    print("Keywords:", keywords)
    for pid, s in scores[:k]:
        print("  %.3f  %s" % (s, prod_map[pid]["name"]))

get_by_keyword(["noi","nau"]); print()
get_by_keyword(["may","xay","ep"])


## 8. Visualize — Products + Users in embedding space

In [ ]:
all_embs = np.vstack([emb_mat] + [user_embeddings[u] for u in sorted(user_embeddings)])
pca = PCA(n_components=2)
coords = pca.fit_transform(all_embs)
n_prod = len(PRODUCTS)
prod_xy = coords[:n_prod]
user_xy = coords[n_prod:]

cats = list({p["cat"] for p in PRODUCTS})
cmap = {c: plt.cm.tab10(i/len(cats)) for i,c in enumerate(cats)}
fig, ax = plt.subplots(figsize=(11,7))
for i,p in enumerate(PRODUCTS):
    ax.scatter(*prod_xy[i], color=cmap[p["cat"]], s=100, zorder=3)
    ax.annotate(p["name"][:20], prod_xy[i], fontsize=7.5, ha="left", va="bottom")
for i,uid in enumerate(sorted(user_embeddings)):
    ax.scatter(*user_xy[i], color="black", s=220, marker="*", zorder=4)
    ax.annotate(uid, user_xy[i], fontsize=10, fontweight="bold", ha="right")
ax.set_title("Products + Users (TF-IDF, PCA 2D)\nStars=users, circles=products")
plt.tight_layout(); plt.show()


## 9. Export pkl

In [ ]:
OUT = "../app/data/embeddings"
os.makedirs(OUT, exist_ok=True)

with open(os.path.join(OUT,"product_embeddings_sample.pkl"),"wb") as f:
    pickle.dump(product_embeddings, f)
with open(os.path.join(OUT,"user_embeddings_sample.pkl"),"wb") as f:
    pickle.dump(user_embeddings, f)

print("Saved product_embeddings:", len(product_embeddings))
print("Saved user_embeddings:", len(user_embeddings))
